# Hyperparameter Tuning, Optimization, and AutoML

Notebook for **Chapter 11** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch11.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
import numpy as np, pandas as pd
def make_churn(n=4000, seed=0):
    rng = np.random.default_rng(seed)
    tenure = rng.integers(1, 60, n); spend = rng.normal(50, 15, n).round(1)
    calls = rng.poisson(1.5, n)
    plan = rng.choice(['basic','plus','pro'], n, p=[.5,.3,.2])
    region = rng.choice(['N','S','E','W'], n)
    z = -0.2 - 0.05*tenure + 0.03*(spend-50) + 0.3*calls + (plan=='basic')*0.6
    z += ((tenure < 12) & (calls >= 3)) * 2.5
    y = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
    y = np.where(rng.random(n) < 0.05, 1-y, y)
    df = pd.DataFrame(dict(tenure=tenure, spend=spend, calls=calls, plan=plan, region=region))
    X = pd.get_dummies(df, columns=['plan','region'], dtype=float)
    X['new_calls'] = ((tenure < 12) & (calls >= 3)).astype(float)
    return X, y

## 11.1 Parameters, Hyperparameters, and the Search Space

In [ ]:
from scipy import stats
print(np.sort(stats.loguniform(0.005, 0.3).rvs(8, random_state=0)).round(4))
print(np.sort(stats.uniform(0.005, 0.295).rvs(8, random_state=0)).round(4))

[0.0283 0.03   0.0465 0.0473 0.059  0.0704 0.0935 0.1926]
[0.13   0.1341 0.1657 0.1669 0.1828 0.1955 0.216  0.2681]


## 11.2 Grid Search and Random Search

In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import (StratifiedKFold, train_test_split, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV, HalvingRandomSearchCV)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
X, y = make_churn()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
base = HistGradientBoostingClassifier(random_state=0)
print("default cv", cross_val_score(base, Xtr, ytr, cv=cv, scoring='roc_auc').mean().round(4))
print("default test", round(roc_auc_score(yte, base.fit(Xtr, ytr).predict_proba(Xte)[:, 1]), 4))

default cv 0.6827
default test 0.7135


In [ ]:
grid = GridSearchCV(HistGradientBoostingClassifier(random_state=0, max_iter=100),
                    {'learning_rate': [0.01, 0.1, 0.3], 'max_leaf_nodes': [4, 16, 64]},
                    cv=cv, scoring='roc_auc').fit(Xtr, ytr)
print(grid.best_params_, grid.best_score_.round(4))
import pandas as pd
r = pd.DataFrame(grid.cv_results_)
print(r.pivot(index='param_max_leaf_nodes', columns='param_learning_rate',
              values='mean_test_score').round(4))

{'learning_rate': 0.1, 'max_leaf_nodes': 4} 0.7286
param_learning_rate     0.01    0.10    0.30
param_max_leaf_nodes                        
4                     0.7148  0.7286  0.7139
16                    0.7188  0.7066  0.6725
64                    0.7009  0.6730  0.6622


In [ ]:
dist = {'learning_rate': stats.loguniform(0.005, 0.3), 'max_leaf_nodes': stats.randint(3, 40),
        'min_samples_leaf': stats.randint(10, 200), 'l2_regularization': stats.loguniform(1e-3, 10),
        'max_iter': stats.randint(30, 300)}
rs = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=0), dist, n_iter=40,
                        cv=cv, scoring='roc_auc', random_state=0).fit(Xtr, ytr)
print("best cv", rs.best_score_.round(4))
print("test", round(roc_auc_score(yte, rs.best_estimator_.predict_proba(Xte)[:, 1]), 4))

best cv 0.729
test 0.7279


## 11.3 Smarter Searches: Successive Halving and Bayesian Ideas

In [ ]:
hs = HalvingRandomSearchCV(HistGradientBoostingClassifier(random_state=0), dist, cv=cv,
        scoring='roc_auc', random_state=0, factor=3, resource='n_samples',
        min_resources=300, n_candidates=40).fit(Xtr, ytr)
print(hs.best_score_.round(4), hs.n_iterations_, hs.n_candidates_, hs.n_resources_)

0.7255 3 [40, 14, 5] [300, 900, 2700]


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
def f(x): return np.sin(3*x) + 0.5*x - 0.2*x**2
grid_x = np.linspace(0, 5, 501)
xs = [0.4, 1.9, 3.3, 4.7]; ys = [f(v) for v in xs]
for it in range(6):
    gp = GaussianProcessRegressor(Matern(length_scale=0.6, nu=2.5), alpha=1e-6,
                                  optimizer=None, normalize_y=True).fit(np.array(xs)[:, None], ys)
    mu, sd = gp.predict(grid_x[:, None], return_std=True); best = max(ys)
    zz = (mu - best) / np.maximum(sd, 1e-9)
    ei = (mu - best) * stats.norm.cdf(zz) + sd * stats.norm.pdf(zz)   # expected improvement
    xn = grid_x[ei.argmax()]; xs.append(float(xn)); ys.append(float(f(xn)))
    print(it, "next x", round(float(xn), 2), "f", round(float(f(xn)), 3), "best", round(max(ys), 3))

0 next x 0.09 f 0.31 best so far 1.1
1 next x 0.66 f 1.16 best so far 1.16
2 next x 0.54 f 1.21 best so far 1.21
3 next x 1.13 f 0.064 best so far 1.21
4 next x 0.57 f 1.21 best so far 1.21
5 next x 2.53 f 0.95 best so far 1.21


In [ ]:
import optuna
def objective(trial):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 3, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 200),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-3, 10, log=True),
    )
    model = HistGradientBoostingClassifier(random_state=0, **params)
    return cross_val_score(model, Xtr, ytr, cv=cv, scoring="roc_auc").mean()

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=40)
print(study.best_params, study.best_value)

## 11.4 Honest Evaluation of Tuning: Nested CV, Over-tuning, Budgets

In [ ]:
from sklearn.tree import DecisionTreeClassifier
rng = np.random.default_rng(0)
Xn = rng.normal(size=(300, 10)); yn = rng.integers(0, 2, 300)
Xa, Xb, ya, yb = train_test_split(Xn, yn, test_size=0.5, random_state=0)
res = []
for i in range(100):
    m = DecisionTreeClassifier(max_depth=int(rng.integers(1, 8)), min_samples_leaf=int(rng.integers(1, 30)),
                               max_features=float(rng.uniform(.2, 1)), random_state=int(rng.integers(0, 1000))).fit(Xa, ya)
    res.append(roc_auc_score(yb, m.predict_proba(Xb)[:, 1]))
print("mean val AUC", np.mean(res).round(3), "best of 100", np.max(res).round(3))

mean val AUC 0.477 best of 100 0.577


In [ ]:
outer = StratifiedKFold(5, shuffle=True, random_state=1)
inner_scores, outer_scores = [], []
for tri, tei in outer.split(X, y):
    s = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=0), dist, n_iter=20,
            cv=StratifiedKFold(5, shuffle=True, random_state=2), scoring='roc_auc',
            random_state=0).fit(X.iloc[tri], y[tri])
    inner_scores.append(s.best_score_)
    outer_scores.append(roc_auc_score(y[tei], s.predict_proba(X.iloc[tei])[:, 1]))
print("inner", np.round(inner_scores, 4), np.mean(inner_scores).round(4))
print("outer", np.round(outer_scores, 4), np.mean(outer_scores).round(4))

inner [0.7247 0.7333 0.7301 0.7333 0.7325] 0.7308
outer [0.7535 0.7192 0.7474 0.7244 0.7383] 0.7366


## 11.5 Optimization Recap: What Training Itself Does

In [ ]:
A = np.diag([1.0, 25.0])
def run(opt, steps=60, lr=0.03):
    w = np.array([4.0, 1.0]); m = np.zeros(2); v = np.zeros(2); vel = np.zeros(2); out = []
    for t in range(1, steps + 1):
        g = A @ w
        if opt == 'gd':   w = w - lr * g
        elif opt == 'mom': vel = 0.9 * vel + g; w = w - lr * vel
        elif opt == 'adam':
            m = 0.9*m + 0.1*g; v = 0.999*v + 0.001*g*g
            mh = m/(1 - 0.9**t); vh = v/(1 - 0.999**t)
            w = w - 0.3 * mh / (np.sqrt(vh) + 1e-8)
        out.append(0.5 * w @ A @ w)
    return out
for o in ['gd', 'mom', 'adam']:
    L = run(o); print(o, "loss at step 10:", round(L[9], 4), "step 30:", round(L[29], 6), "step 60:", round(L[59], 8))
for lr in [0.01, 0.03, 0.07, 0.085]:
    print("gd lr", lr, "loss@60", run('gd', lr=lr)[59])

gd loss at step 10: 4.3504 step 30: 1.286453 step 60: 0.20687028
mom loss at step 10: 5.0254 step 30: 0.591473 step 60: 0.01943896
adam loss at step 10: 2.2256 step 30: 0.47766 step 60: 0.00240678
gd lr 0.01 loss@60 2.395043130498667
gd lr 0.03 loss@60 0.20687027841451988
gd lr 0.07 loss@60 0.0013214289388152583
gd lr 0.085 loss@60 17187499.822868153


## 11.6 AutoML: What It Does and When to Use It

In [ ]:
from flaml import AutoML
automl = AutoML()
automl.fit(X_train=Xtr, y_train=ytr, task="classification", metric="roc_auc", time_budget=60)
print(automl.best_estimator, automl.best_config)

## Chapter recap

## Practice set

Problem 11.P.1 (easy) Are `max_depth` and the leaf values of a tree parameters or hyperparameters?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch11.html